In [1]:
import re
from tqdm.notebook import tqdm
import multiprocessing as mp
from functools import partial
import os
import numpy as np
import itertools

import time
import humanize
import psutil

In [2]:
def load_keywords(keywords_file):
    """
    Load keywords from a file or use a predefined list
    """
    with open(keywords_file, 'r', encoding='utf-8') as f:

        # Read keywords one per line and strip whitespace
        keywords = []
        for line in f:
            line=line.strip()
            keywords.append(line)
    print(f"Loaded {len(keywords)} keywords from {keywords_file}")
    print(f"Using {len(keywords)} predefined keywords")
    
    return keywords

  # Load keywords

    

In [3]:
keywords_pattern = None

def check_tweet_bytes(tweet_bytes):
    if keywords_pattern.search(tweet_bytes):
        return tweet_bytes
    return None

def search_tweets_raw(input_file, output_file, keywords_file, chunk_size=10000000, mp_chunksize=50000):
    total_processed = 0
    total_matches = 0
    
    keywords = load_keywords(keywords_file)
    # Compile the regex pattern for bytes
    global keywords_pattern
    bytes_keywords = [kw.lower().encode('utf-8') for kw in keywords]
    pattern_string = b'|'.join(b'\\b' + re.escape(kw) + b'\\b' for kw in bytes_keywords)
    keywords_pattern = re.compile(pattern_string, re.IGNORECASE)
    
    with open(output_file, 'wb') as out_file:
        with mp.Pool(processes=mp.cpu_count()) as pool:
            with open(input_file, 'rb') as in_file:
                pbar = tqdm(desc="Processing file", unit="lines")
                
                while True:
                    # Read a chunk of lines
                    chunk = list(itertools.islice(in_file, chunk_size))
                    if not chunk:
                        break
                    
                    chunk_len = len(chunk)
                    total_processed += chunk_len
                    
                    # Process the chunk with multiprocessing
                    chunk_results = pool.imap(check_tweet_bytes, chunk, chunksize=mp_chunksize)
                    
                    # Track progress with tqdm for this chunk
                    chunk_matches = 0
                    for result in tqdm(chunk_results, total=chunk_len, desc=f"Chunk {total_processed//chunk_size}", leave=False):
                        if result is not None:
                            out_file.write(result)
                            chunk_matches += 1
                    
                    total_matches += chunk_matches
                    pbar.update(chunk_len)
                    pbar.set_postfix(matches=total_matches)
                
                pbar.close()
    
    print(f"Completed: {total_processed} lines processed, {total_matches} matches found ({total_matches/total_processed:.2%})")


In [ ]:
 # Example usage/USA
input_file = 'query_results/En_Nov_Feb_2021.txt'
output_file = 'keyword_filter_output/AU_Nov_Feb_2021_.txt'
keywords_file = 'AU/Keyword_AU.txt'  

# Run search with keywords from file
start_time = time.time()
search_tweets_raw(input_file, output_file, keywords_file)
print("Time taken: {}".format(humanize.precisedelta(time.time() - start_time, suppress=['days', 'milliseconds', 'microseconds'])))
print("Memory used: {}".format(humanize.naturalsize(psutil.Process().memory_info().rss)))


In [ ]:
 # Example usage/AU
input_file = 'query_results/All_June_Nov2020.txt'
output_file = 'keyword_filter_output/AU_June-Nov2020_.txt'
keywords_file = 'Keyword_AU.txt'

# Run search with keywords from file
start_time = time.time()
search_tweets_raw(input_file, output_file, keywords_file)
print("Time taken: {}".format(humanize.precisedelta(time.time() - start_time, suppress=['days', 'milliseconds', 'microseconds'])))
print("Memory used: {}".format(humanize.naturalsize(psutil.Process().memory_info().rss)))
                

In [11]:
 # Example usage/ARG
input_file = 'ARG/ARG_March-June2020.txt'
output_file = 'ARG/arg_refiltered__March-June2020.txt'
keywords_file = 'ARG/ARG Keywords.txt'  



# Run search with keywords from file
start_time = time.time()
search_tweets_raw(input_file, output_file, keywords_file)
print("Time taken: {}".format(humanize.precisedelta(time.time() - start_time, suppress=['days', 'milliseconds', 'microseconds'])))
print("Memory used: {}".format(humanize.naturalsize(psutil.Process().memory_info().rss)))

Loaded 114 keywords from ARG/ARG Keywords.txt
Using 114 predefined keywords


Processing file: 0lines [00:00, ?lines/s]

Chunk 1:   0%|          | 0/10000000 [00:00<?, ?it/s]

Chunk 1:   0%|          | 0/4164233 [00:01<?, ?it/s]

Completed: 14164233 lines processed, 4250258 matches found (30.01%)
Time taken: 4 minutes and 10 seconds
Memory used: 22.7 GB
